# Evidencia de Aprendizaje 1 — Analizando datos de comercio electrónico con Web Scraping

**Estudiantes:**
- Rosero Carrera, Samuel
- Rico, Brian

**Institución:** *(completar)*
**Curso:** *(completar)*
**Docente:** *(completar)*
**Fecha de entrega:** *(completar)*

---

**Producto analizado:** *Cien años de soledad* (Edición conmemorativa de la RAE y la ASALE) — Gabriel García Márquez
**Sitio web:** [Buscalibre Colombia](https://www.buscalibre.com.co)
**URL del producto:** https://www.buscalibre.com.co/libro-cien-anos-de-soledad-edicion-conmemorativa-de-la-rae-y-la-asale/9789585118010/p/52342100
**Fecha de consulta:** *(completar con la fecha en que se ejecute el notebook)*

> **Nota:** Los precios, disponibilidad y opiniones pueden cambiar con el tiempo. Los datos que se muestran corresponden al momento de ejecución.

## 1. Objetivo

**Objetivo general:** Aplicar la metodología de web scraping para extraer y comparar información pública del producto *Cien años de soledad* en Buscalibre Colombia, utilizando tres herramientas de Python (BeautifulSoup, Selenium y Scrapy), y evaluar sus diferencias en alcance, facilidad de uso y completitud de los datos obtenidos.

**Objetivos específicos:**

1. Extraer campos públicos del producto (título, autor, ISBN, precio, disponibilidad, entre otros) empleando cada uno de los tres métodos.
2. Comparar los resultados obtenidos por cada método e identificar diferencias, ventajas y limitaciones.
3. Definir y calcular métricas de éxito que permitan evaluar la efectividad de cada herramienta.
4. Verificar el cumplimiento de las condiciones de acceso del sitio (`robots.txt`).

## 2. Campos objetivo y métricas de éxito

Se intentará extraer los siguientes campos del producto. No todos estarán disponibles con cada método, y eso forma parte del análisis comparativo.

| # | Campo | Descripción |
|---|-------|-------------|
| 1 | Título | Nombre completo del libro |
| 2 | Autor | Autor(es) del libro |
| 3 | ISBN | Identificador ISBN-13 |
| 4 | Precio actual | Precio de venta en COP |
| 5 | Precio original | Precio sin descuento |
| 6 | Descuento | Porcentaje de descuento |
| 7 | Disponibilidad | Estado de stock |
| 8 | Editorial | Casa editorial |
| 9 | N.º de páginas | Cantidad de páginas |
| 10 | Idioma | Idioma del libro |

**Métricas de éxito:**

- **Completitud:** Campos obtenidos / Campos objetivo (10). Se calcula por método.
- **Concordancia:** Proporción de campos donde al menos dos métodos obtienen el mismo valor.
- **Ejecución sin errores:** Si el método se ejecutó sin excepciones no controladas (Sí/No).

## 3. Instalación de dependencias

Se instalan las bibliotecas necesarias para los tres métodos de scraping. `requests` y `beautifulsoup4` son para el método estático; `selenium` para el método con navegador; y `scrapy` para el framework de crawling. `lxml` mejora el rendimiento del parser HTML.

En Google Colab, Selenium requiere además la instalación de Chromium y su driver.

In [1]:
!pip install requests beautifulsoup4 lxml selenium scrapy -q


[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## 4. Configuración general

Se definen las constantes del análisis: la URL del producto, los headers HTTP para simular un navegador convencional y la lista de campos que se intentará extraer. Usar un User-Agent real es una práctica estándar que evita bloqueos innecesarios sin evadir mecanismos de seguridad.

In [2]:
import requests
from bs4 import BeautifulSoup
import json
import re
import time
import warnings
from datetime import datetime

import pandas as pd

warnings.filterwarnings("ignore")

URL_PRODUCTO = (
    "https://www.buscalibre.com.co/libro-cien-anos-de-soledad-"
    "edicion-conmemorativa-de-la-rae-y-la-asale/9789585118010/p/52342100"
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "es-CO,es;q=0.9",
}

CAMPOS_OBJETIVO = [
    "titulo", "autor", "isbn", "precio_actual", "precio_original",
    "descuento", "disponibilidad", "editorial", "num_paginas", "idioma",
]

FECHA_CONSULTA = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Fecha de consulta: {FECHA_CONSULTA}")
print(f"URL: {URL_PRODUCTO}")

Fecha de consulta: 2026-10-19 21:54:08
URL: https://www.buscalibre.com.co/libro-cien-anos-de-soledad-edicion-conmemorativa-de-la-rae-y-la-asale/9789585118010/p/52342100


## 5. Revisión de robots.txt

Antes de realizar cualquier solicitud, se verifica el archivo `robots.txt` del sitio para asegurar que el scraping de fichas de producto está permitido. Esto es una buena práctica ética y legal.

Según el `robots.txt` de Buscalibre Colombia (consultado previamente):
- Las fichas de producto (`/libro-...`) **no están bloqueadas**.
- Las rutas bloqueadas son: carrito (`/carro/`), pagos (`/pagar/`), área de usuario (`/u/`) y búsquedas con parámetros.
- No se define un `Crawl-delay`.
- Bots como `ClaudeBot`, `ChatGPT-User` y `Google-Extended` están **explícitamente permitidos**.

Se realiza solo **una solicitud por método** a una única página de producto, lo cual es un uso mínimo y respetuoso del sitio.

In [3]:
robots_url = "https://www.buscalibre.com.co/robots.txt"
try:
    r = requests.get(robots_url, headers=HEADERS, timeout=10)
    print(f"Status: {r.status_code}")
    # Mostrar solo las primeras líneas relevantes
    lineas = r.text.split("\n")
    for linea in lineas[:30]:
        print(linea)
    print("\n... (archivo completo disponible en la URL)")
except Exception as e:
    print(f"No se pudo obtener robots.txt: {e}")

Status: 200
# OpenAI
User-agent: GPTBot
Allow: /
Disallow: /carro/
Disallow: /*/carro/
Disallow: /v2/carro/
Disallow: /pagar/*
Disallow: /u/
Disallow: /u/*

User-agent: OAI-SearchBot
Allow: /

User-agent: ChatGPT-User
Allow: /

# Anthropic (Claude)
User-agent: ClaudeBot
Allow: /
Disallow: /carro/
Disallow: /*/carro/
Disallow: /v2/carro/
Disallow: /pagar/*
Disallow: /u/
Disallow: /u/*

User-agent: Claude-SearchBot
Allow: /

User-agent: Claude-User

... (archivo completo disponible en la URL)


## 6. Método 1 — BeautifulSoup

### ¿Qué es BeautifulSoup?

BeautifulSoup es una biblioteca de Python para extraer datos de archivos HTML y XML. Funciona sobre el HTML **estático** que devuelve el servidor ante una solicitud HTTP convencional (mediante `requests`). No ejecuta JavaScript, por lo que solo accede al contenido presente en el HTML inicial.

### ¿Por qué se usa aquí?

Es el método más sencillo y ligero. No requiere navegador ni dependencias pesadas. Es ideal cuando la información buscada está en el HTML inicial, en meta-etiquetas Open Graph o en datos estructurados JSON-LD.

### Estrategia de extracción

1. Se obtiene el título desde `<p class="tituloProducto">` o desde el `<h1>`.
2. Se extraen precios desde los elementos `.pvp` (precio original) y `.ped` (precio con descuento), y desde el bloque JSON-LD.
3. La ficha técnica de Buscalibre usa una estructura `div.ficha > div.row` con columnas `col-xs-5` (etiqueta) y `col-xs-7` (valor), además de IDs como `#metadata-idioma`, `#metadata-Autor`, etc.
4. Se complementa con datos estructurados JSON-LD (`<script type="application/ld+json">`) y el ISBN presente en la URL.

In [4]:
def extraer_beautifulsoup(url, headers):
    """Extrae información del producto usando BeautifulSoup sobre HTML estático."""
    datos = {campo: None for campo in CAMPOS_OBJETIVO}

    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()
    print(f"Status HTTP: {response.status_code}")

    soup = BeautifulSoup(response.text, "lxml")

    # --- Título desde elementos HTML ---
    titulo_el = soup.select_one("p.tituloProducto")
    if titulo_el:
        datos["titulo"] = titulo_el.get_text(strip=True)
    else:
        h1 = soup.select_one("h1")
        if h1:
            datos["titulo"] = h1.get_text(strip=True)

    # --- JSON-LD (datos estructurados) ---
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            ld = json.loads(script.string)
            if isinstance(ld, list):
                ld = ld[0]
            tipo = ld.get("@type", "")
            es_producto = tipo == "Product" or "Product" in tipo if isinstance(tipo, list) else tipo in ("Product", "Book")
            if es_producto:
                datos["titulo"] = datos["titulo"] or ld.get("name")
                ofertas = ld.get("offers", {})
                if isinstance(ofertas, list):
                    ofertas = ofertas[0] if ofertas else {}
                precio_ld = ofertas.get("price")
                if precio_ld:
                    datos["precio_actual"] = str(precio_ld)
                avail = ofertas.get("availability", "")
                if avail:
                    datos["disponibilidad"] = avail.replace("http://schema.org/", "").replace("https://schema.org/", "")
        except (json.JSONDecodeError, AttributeError, TypeError):
            continue

    # --- Precios desde selectores CSS (.pvp = original, .ped = actual) ---
    pvp_el = soup.select_one(".colPrecio .pvp, .pvp")
    if pvp_el:
        datos["precio_original"] = pvp_el.get_text(strip=True)

    ped_el = soup.select_one(".colPrecio .ped, .ped")
    if ped_el:
        datos["precio_actual"] = ped_el.get_text(strip=True)

    # --- Descuento: calcular a partir de los precios o buscar en opcionPrecio ---
    opcion = soup.select_one(".opcionPrecio.selected")
    if opcion:
        texto_opcion = opcion.get_text()
        desc_match = re.search(r"-(\d+)%", texto_opcion)
        if desc_match:
            datos["descuento"] = f"-{desc_match.group(1)}%"

    # --- Disponibilidad desde .stock ---
    stock_el = soup.select_one("p.stock, .stock")
    if stock_el:
        datos["disponibilidad"] = stock_el.get_text(strip=True)

    # --- Ficha técnica: div.ficha > div.row (col-xs-5 = label, col-xs-7 = valor) ---
    ficha = soup.select_one("div.ficha")
    if ficha:
        for row in ficha.select("div.row"):
            label_el = row.select_one(".col-xs-5")
            value_el = row.select_one(".col-xs-7")
            if label_el and value_el:
                label = label_el.get_text(strip=True).lower()
                value = value_el.get_text(strip=True)
                if "editorial" in label or "sello" in label:
                    datos["editorial"] = value
                elif "página" in label:
                    datos["num_paginas"] = value
                elif label == "idioma":
                    datos["idioma"] = value
                elif label == "isbn" and not datos["isbn"]:
                    datos["isbn"] = value
                elif "autor" in label and not datos["autor"]:
                    datos["autor"] = value

    # --- Autor desde metadatos o colaborador ---
    if not datos["autor"]:
        autor_el = soup.select_one("#metadata-Autor a, div.div-colaborador-name-heading")
        if autor_el:
            datos["autor"] = autor_el.get_text(strip=True)

    # --- ISBN desde la URL como respaldo ---
    if not datos["isbn"]:
        isbn_match = re.search(r"/(\d{13})/", url)
        if isbn_match:
            datos["isbn"] = isbn_match.group(1)

    return datos

print("Ejecutando BeautifulSoup...\n")
try:
    resultados_bs4 = extraer_beautifulsoup(URL_PRODUCTO, HEADERS)
    print("Campos extraídos:")
    for campo, valor in resultados_bs4.items():
        estado = "✓" if valor else "✗"
        print(f"  {estado} {campo}: {valor}")
    exito_bs4 = True
except Exception as e:
    print(f"Error: {e}")
    resultados_bs4 = {campo: None for campo in CAMPOS_OBJETIVO}
    exito_bs4 = False

Ejecutando BeautifulSoup...



Status HTTP: 200
Campos extraídos:
  ✓ titulo: Cien años de soledad (Edición conmemorativa de la RAE y la ASALE)
  ✓ autor: Gabriel García Márquez
  ✓ isbn: 9789585118010
  ✓ precio_actual: $ 64.350
  ✓ precio_original: $ 99.000
  ✓ descuento: -35%
  ✓ disponibilidad: Quedan 18 unidades
  ✗ editorial: None
  ✓ num_paginas: 614
  ✓ idioma: Español


## 7. Método 2 — Selenium

### ¿Qué es Selenium?

Selenium es una herramienta que automatiza navegadores web. A diferencia de BeautifulSoup, Selenium **ejecuta JavaScript**, lo que permite acceder a contenido que se carga dinámicamente después de la solicitud inicial. Controla un navegador real (Chrome, Firefox, etc.) de forma programática.

### ¿Por qué se usa aquí?

Buscalibre carga parte de la información del producto mediante JavaScript (ficha técnica, opiniones, etc.). Selenium permite esperar a que estos elementos aparezcan en el DOM y luego extraerlos. Es más lento y pesado que BeautifulSoup, pero accede a más contenido.

### Configuración

- **Local (Jupyter):** Requiere tener Chrome y ChromeDriver instalados.
- **Google Colab:** Se instala Chromium automáticamente con los comandos indicados.

Si Selenium no está disponible en el entorno actual, el código captura el error y continúa.

In [5]:
import os
import sys

def configurar_selenium():
    """Configura Selenium según el entorno (Colab o local)."""
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service

    opciones = Options()
    opciones.add_argument("--headless")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument(f"user-agent={HEADERS['User-Agent']}")

    en_colab = "google.colab" in sys.modules

    if en_colab:
        os.system("apt-get update -qq && apt-get install -y -qq chromium-chromedriver > /dev/null 2>&1")
        opciones.binary_location = "/usr/bin/chromium-browser"
        service = Service("/usr/bin/chromedriver")
        driver = webdriver.Chrome(service=service, options=opciones)
    else:
        driver = webdriver.Chrome(options=opciones)

    return driver


def extraer_selenium(url):
    """Extrae información del producto usando Selenium con navegador headless."""
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC

    datos = {campo: None for campo in CAMPOS_OBJETIVO}
    driver = configurar_selenium()

    try:
        driver.get(url)
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "p.tituloProducto, h1"))
        )
        time.sleep(3)

        # --- Título ---
        try:
            el = driver.find_element(By.CSS_SELECTOR, "p.tituloProducto")
            datos["titulo"] = el.text.strip()
        except Exception:
            try:
                el = driver.find_element(By.TAG_NAME, "h1")
                datos["titulo"] = el.text.strip()
            except Exception:
                pass

        # --- Autor desde ficha o colaborador ---
        try:
            el = driver.find_element(By.CSS_SELECTOR, "#metadata-Autor a, div.div-colaborador-name-heading")
            datos["autor"] = el.text.strip()
        except Exception:
            pass

        # --- Precio actual (.ped) ---
        try:
            el = driver.find_element(By.CSS_SELECTOR, ".colPrecio .ped, .ped")
            datos["precio_actual"] = el.text.strip()
        except Exception:
            pass

        # --- Precio original (.pvp) ---
        try:
            el = driver.find_element(By.CSS_SELECTOR, ".colPrecio .pvp, .pvp")
            datos["precio_original"] = el.text.strip()
        except Exception:
            pass

        # --- Descuento ---
        try:
            opcion = driver.find_element(By.CSS_SELECTOR, ".opcionPrecio.selected")
            texto = opcion.text
            desc_match = re.search(r"-(\d+)%", texto)
            if desc_match:
                datos["descuento"] = f"-{desc_match.group(1)}%"
        except Exception:
            pass

        # --- Disponibilidad ---
        try:
            el = driver.find_element(By.CSS_SELECTOR, "p.stock, .stock")
            datos["disponibilidad"] = el.text.strip()
        except Exception:
            pass

        # --- Ficha técnica: div.ficha > div.row ---
        try:
            ficha = driver.find_element(By.CSS_SELECTOR, "div.ficha")
            filas = ficha.find_elements(By.CSS_SELECTOR, "div.row")
            for fila in filas:
                try:
                    label_el = fila.find_element(By.CSS_SELECTOR, ".col-xs-5")
                    value_el = fila.find_element(By.CSS_SELECTOR, ".col-xs-7")
                    label = label_el.text.strip().lower()
                    value = value_el.text.strip()
                    if not value:
                        continue
                    if "editorial" in label or "sello" in label:
                        datos["editorial"] = value
                    elif "página" in label:
                        datos["num_paginas"] = value
                    elif label == "idioma":
                        datos["idioma"] = value
                    elif label == "isbn" and not datos["isbn"]:
                        datos["isbn"] = value
                    elif "autor" in label and not datos["autor"]:
                        datos["autor"] = value
                except Exception:
                    continue
        except Exception:
            pass

        # --- ISBN desde URL como respaldo ---
        if not datos["isbn"]:
            isbn_match = re.search(r"/(\d{13})/", url)
            if isbn_match:
                datos["isbn"] = isbn_match.group(1)

        # --- Complementar con JSON-LD del page source ---
        try:
            soup = BeautifulSoup(driver.page_source, "lxml")
            for script in soup.find_all("script", type="application/ld+json"):
                try:
                    ld = json.loads(script.string)
                    if isinstance(ld, list):
                        ld = ld[0]
                    tipo = ld.get("@type", "")
                    es_producto = "Product" in str(tipo)
                    if es_producto:
                        datos["titulo"] = datos["titulo"] or ld.get("name")
                        ofertas = ld.get("offers", {})
                        if isinstance(ofertas, list):
                            ofertas = ofertas[0] if ofertas else {}
                        if not datos["precio_actual"] and ofertas.get("price"):
                            datos["precio_actual"] = str(ofertas["price"])
                except (json.JSONDecodeError, AttributeError):
                    continue
        except Exception:
            pass

    finally:
        driver.quit()

    return datos


print("Ejecutando Selenium...\n")
try:
    resultados_selenium = extraer_selenium(URL_PRODUCTO)
    print("Campos extraídos:")
    for campo, valor in resultados_selenium.items():
        estado = "✓" if valor else "✗"
        print(f"  {estado} {campo}: {valor}")
    exito_selenium = True
except Exception as e:
    print(f"Error al ejecutar Selenium: {e}")
    print("Selenium requiere un navegador Chrome/Chromium instalado.")
    print("En Google Colab se configura automáticamente.")
    resultados_selenium = {campo: None for campo in CAMPOS_OBJETIVO}
    exito_selenium = False

Ejecutando Selenium...



Campos extraídos:
  ✓ titulo: Cien años de soledad (Edición conmemorativa de la RAE y la ASALE)
  ✓ autor: Gabriel García Márquez
  ✓ isbn: 9789585118010
  ✓ precio_actual: $ 64.350
  ✓ precio_original: $ 99.000
  ✓ descuento: -35%
  ✗ disponibilidad: 
  ✗ editorial: None
  ✓ num_paginas: 614
  ✓ idioma: Español


## 8. Método 3 — Scrapy

### ¿Qué es Scrapy?

Scrapy es un framework completo de web scraping y crawling en Python. Está diseñado para extraer datos de sitios web de forma eficiente y escalable. A diferencia de BeautifulSoup (que solo parsea HTML), Scrapy gestiona solicitudes HTTP, concurrencia, políticas de cortesía, pipelines de datos y más.

### ¿Por qué se usa aquí?

Scrapy permite definir un "spider" con reglas de extracción declarativas. Aunque para una sola página su potencia es excesiva, demuestra cómo se estructura un proyecto de scraping profesional. Al igual que BeautifulSoup, trabaja sobre HTML estático (no ejecuta JavaScript).

### Ejecución en notebooks

Scrapy usa Twisted (un framework asíncrono) que puede entrar en conflicto con el event loop de Jupyter/Colab. Para evitar problemas, se escribe el spider en un archivo `.py` y se ejecuta como subproceso. Los resultados se guardan en un archivo JSON que luego se lee desde el notebook.

In [6]:
import subprocess
import tempfile

spider_code = '''
import scrapy
import json
import re

class BuscalibreSpider(scrapy.Spider):
    name = "buscalibre"
    start_urls = [
        "https://www.buscalibre.com.co/libro-cien-anos-de-soledad-"
        "edicion-conmemorativa-de-la-rae-y-la-asale/9789585118010/p/52342100"
    ]

    custom_settings = {
        "USER_AGENT": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "ROBOTSTXT_OBEY": True,
        "LOG_LEVEL": "WARNING",
        "FEEDS": {},
    }

    def parse(self, response):
        datos = {
            "titulo": None, "autor": None, "isbn": None,
            "precio_actual": None, "precio_original": None,
            "descuento": None, "disponibilidad": None,
            "editorial": None, "num_paginas": None, "idioma": None,
        }

        # --- Título ---
        titulo = response.css("p.tituloProducto::text").get()
        if titulo:
            datos["titulo"] = titulo.strip()
        else:
            h1 = response.css("h1::text").get()
            if h1:
                datos["titulo"] = h1.strip()

        # --- Precios (.pvp = original, .ped = actual) ---
        pvp = response.css(".colPrecio .pvp::text, .pvp::text").get()
        if pvp:
            datos["precio_original"] = pvp.strip()

        ped = response.css(".colPrecio .ped::text, .ped::text").get()
        if ped:
            datos["precio_actual"] = ped.strip()

        # --- Descuento ---
        opcion_texto = response.css(".opcionPrecio.selected::text").getall()
        opcion_texto = " ".join(opcion_texto)
        desc_match = re.search(r"-(\\d+)%", opcion_texto)
        if desc_match:
            datos["descuento"] = f"-{desc_match.group(1)}%"
        else:
            # Buscar en todo el bloque opcionPrecio
            opcion_html = response.css(".opcionPrecio.selected").get() or ""
            desc_match2 = re.search(r"-(\\d+)%", opcion_html)
            if desc_match2:
                datos["descuento"] = f"-{desc_match2.group(1)}%"

        # --- Disponibilidad ---
        stock = response.css("p.stock::text, .stock::text").get()
        if stock:
            datos["disponibilidad"] = stock.strip()

        # --- Ficha técnica: div.ficha > div.row ---
        for row in response.css("div.ficha div.row"):
            label = row.css(".col-xs-5 *::text, .col-xs-5::text").getall()
            label = " ".join(t.strip() for t in label).strip().lower()
            value = row.css(".col-xs-7 *::text, .col-xs-7::text").getall()
            value = " ".join(t.strip() for t in value).strip()
            if not value:
                continue
            if "editorial" in label or "sello" in label:
                datos["editorial"] = value
            elif "página" in label:
                datos["num_paginas"] = value.split()[0] if value else value
            elif label == "idioma":
                datos["idioma"] = value
            elif label == "isbn" and not datos["isbn"]:
                datos["isbn"] = value
            elif "autor" in label and not datos["autor"]:
                # Limpiar texto extra como "Ver Página del Autor"
                valor_limpio = value.split("Ver")[0].strip()
                datos["autor"] = valor_limpio

        # --- JSON-LD ---
        for script in response.css("script[type=\\'application/ld+json\\']::text").getall():
            try:
                ld = json.loads(script)
                if isinstance(ld, list):
                    ld = ld[0]
                if "Product" in str(ld.get("@type", "")):
                    datos["titulo"] = datos["titulo"] or ld.get("name")
                    ofertas = ld.get("offers", {})
                    if isinstance(ofertas, list):
                        ofertas = ofertas[0] if ofertas else {}
                    if not datos["precio_actual"] and ofertas.get("price"):
                        datos["precio_actual"] = str(ofertas["price"])
            except (json.JSONDecodeError, AttributeError):
                continue

        # --- ISBN desde URL ---
        if not datos["isbn"]:
            isbn_match = re.search(r"/(\\d{13})/", response.url)
            if isbn_match:
                datos["isbn"] = isbn_match.group(1)

        yield datos
'''

spider_path = os.path.join(tempfile.gettempdir(), "buscalibre_spider.py")
output_path = os.path.join(tempfile.gettempdir(), "buscalibre_output.json")

# Limpiar salida previa si existe
if os.path.exists(output_path):
    os.remove(output_path)

with open(spider_path, "w") as f:
    f.write(spider_code)

print("Ejecutando Scrapy...\n")
try:
    resultado = subprocess.run(
        [
            sys.executable, "-m", "scrapy", "runspider", spider_path,
            "-o", output_path, "--nolog",
        ],
        capture_output=True, text=True, timeout=30,
    )

    if os.path.exists(output_path) and os.path.getsize(output_path) > 2:
        with open(output_path) as f:
            scrapy_data = json.load(f)
        if scrapy_data:
            resultados_scrapy = scrapy_data[0]
            print("Campos extraídos:")
            for campo, valor in resultados_scrapy.items():
                estado = "✓" if valor else "✗"
                print(f"  {estado} {campo}: {valor}")
            exito_scrapy = True
        else:
            print("Scrapy se ejecutó pero no devolvió datos.")
            resultados_scrapy = {campo: None for campo in CAMPOS_OBJETIVO}
            exito_scrapy = False
    else:
        print("No se generó archivo de salida.")
        if resultado.stderr:
            print(f"Error: {resultado.stderr[:500]}")
        resultados_scrapy = {campo: None for campo in CAMPOS_OBJETIVO}
        exito_scrapy = False

except subprocess.TimeoutExpired:
    print("Scrapy excedió el tiempo de espera.")
    resultados_scrapy = {campo: None for campo in CAMPOS_OBJETIVO}
    exito_scrapy = False
except Exception as e:
    print(f"Error al ejecutar Scrapy: {e}")
    resultados_scrapy = {campo: None for campo in CAMPOS_OBJETIVO}
    exito_scrapy = False
finally:
    for path in [spider_path, output_path]:
        if os.path.exists(path):
            os.remove(path)

Ejecutando Scrapy...



Campos extraídos:
  ✓ titulo: Cien años de soledad (Edición conmemorativa de la RAE y la ASALE)
  ✓ autor: Gabriel García Márquez
  ✓ isbn: 9789585118010
  ✓ precio_actual: $ 64.350
  ✓ precio_original: $ 99.000
  ✓ descuento: -35%
  ✓ disponibilidad: Quedan 18 unidades
  ✗ editorial: None
  ✓ num_paginas: 614
  ✓ idioma: Español


## 9. Comparación de resultados

Se construye una tabla comparativa con los campos extraídos por cada método. Esto permite identificar qué herramienta logró obtener más información y dónde hay coincidencias o discrepancias entre los resultados.

In [7]:
comparacion = pd.DataFrame({
    "Campo": CAMPOS_OBJETIVO,
    "BeautifulSoup": [resultados_bs4.get(c) for c in CAMPOS_OBJETIVO],
    "Selenium": [resultados_selenium.get(c) for c in CAMPOS_OBJETIVO],
    "Scrapy": [resultados_scrapy.get(c) for c in CAMPOS_OBJETIVO],
})

print("\n" + "=" * 80)
print("TABLA COMPARATIVA DE RESULTADOS")
print("=" * 80)
print(comparacion.to_string(index=False))
print()


TABLA COMPARATIVA DE RESULTADOS
          Campo                                                     BeautifulSoup                                                          Selenium                                                            Scrapy
         titulo Cien años de soledad (Edición conmemorativa de la RAE y la ASALE) Cien años de soledad (Edición conmemorativa de la RAE y la ASALE) Cien años de soledad (Edición conmemorativa de la RAE y la ASALE)
          autor                                            Gabriel García Márquez                                            Gabriel García Márquez                                            Gabriel García Márquez
           isbn                                                     9789585118010                                                     9789585118010                                                     9789585118010
  precio_actual                                                          $ 64.350                              

## 10. Métricas de éxito

Se calculan las tres métricas definidas al inicio:

1. **Completitud:** Proporción de campos no nulos sobre el total de campos objetivo (10), calculada por método.
2. **Concordancia:** Proporción de campos donde al menos dos métodos obtienen el mismo valor no nulo.
3. **Ejecución sin errores:** Si cada método se ejecutó sin excepciones no controladas.

In [8]:
total_campos = len(CAMPOS_OBJETIVO)

# Completitud por método
campos_bs4 = sum(1 for c in CAMPOS_OBJETIVO if resultados_bs4.get(c))
campos_sel = sum(1 for c in CAMPOS_OBJETIVO if resultados_selenium.get(c))
campos_scr = sum(1 for c in CAMPOS_OBJETIVO if resultados_scrapy.get(c))

completitud_bs4 = campos_bs4 / total_campos
completitud_sel = campos_sel / total_campos
completitud_scr = campos_scr / total_campos

# Concordancia: campos donde al menos 2 métodos coinciden
concordantes = 0
for campo in CAMPOS_OBJETIVO:
    valores = [
        resultados_bs4.get(campo),
        resultados_selenium.get(campo),
        resultados_scrapy.get(campo),
    ]
    valores_no_nulos = [str(v).strip() for v in valores if v is not None]
    if len(valores_no_nulos) >= 2:
        # Verificar si al menos dos son iguales
        from collections import Counter
        conteo = Counter(valores_no_nulos)
        if conteo.most_common(1)[0][1] >= 2:
            concordantes += 1

concordancia = concordantes / total_campos

print("=" * 60)
print("MÉTRICAS DE ÉXITO")
print("=" * 60)
print(f"\nCompletitud:")
print(f"  BeautifulSoup: {campos_bs4}/{total_campos} = {completitud_bs4:.0%}")
print(f"  Selenium:      {campos_sel}/{total_campos} = {completitud_sel:.0%}")
print(f"  Scrapy:        {campos_scr}/{total_campos} = {completitud_scr:.0%}")
print(f"\nConcordancia entre métodos: {concordantes}/{total_campos} = {concordancia:.0%}")
print(f"\nEjecución sin errores:")
print(f"  BeautifulSoup: {'Sí' if exito_bs4 else 'No'}")
print(f"  Selenium:      {'Sí' if exito_selenium else 'No'}")
print(f"  Scrapy:        {'Sí' if exito_scrapy else 'No'}")

# Tabla resumen de métricas
metricas_df = pd.DataFrame({
    "Métrica": ["Completitud", "Ejecución sin errores"],
    "BeautifulSoup": [f"{completitud_bs4:.0%}", "Sí" if exito_bs4 else "No"],
    "Selenium": [f"{completitud_sel:.0%}", "Sí" if exito_selenium else "No"],
    "Scrapy": [f"{completitud_scr:.0%}", "Sí" if exito_scrapy else "No"],
})
print(f"\nConcordancia general: {concordancia:.0%}")
print()
print(metricas_df.to_string(index=False))

MÉTRICAS DE ÉXITO

Completitud:
  BeautifulSoup: 9/10 = 90%
  Selenium:      8/10 = 80%
  Scrapy:        9/10 = 90%

Concordancia entre métodos: 9/10 = 90%

Ejecución sin errores:
  BeautifulSoup: Sí
  Selenium:      Sí
  Scrapy:        Sí

Concordancia general: 90%

              Métrica BeautifulSoup Selenium Scrapy
          Completitud           90%      80%    90%
Ejecución sin errores            Sí       Sí     Sí


## 11. Análisis comparativo de los tres métodos

| Aspecto | BeautifulSoup | Selenium | Scrapy |
|---------|--------------|----------|--------|
| **Tipo de acceso** | HTML estático (requests) | Navegador completo (ejecuta JS) | HTML estático (framework propio) |
| **Velocidad** | Muy rápida | Lenta (carga navegador) | Rápida |
| **Dependencias** | Ligeras (requests, bs4) | Pesadas (Chrome, ChromeDriver) | Moderadas (Twisted, lxml) |
| **Contenido dinámico** | No accede | Sí accede | No accede |
| **Escalabilidad** | Manual | Manual | Alta (concurrencia, pipelines) |
| **Uso en Colab** | Sin problemas | Requiere instalar Chromium | Puede tener conflictos con el event loop |
| **Robots.txt** | Debe verificarse manualmente | Debe verificarse manualmente | Lo respeta por defecto |

### Diferencias esperadas

- **BeautifulSoup y Scrapy** deberían obtener resultados similares, ya que ambos trabajan sobre el HTML estático. Scrapy además respeta automáticamente `robots.txt`.
- **Selenium** tiene la ventaja de ejecutar JavaScript, lo que le permite acceder a contenido cargado dinámicamente (ficha técnica, opiniones, precios actualizados via AJAX).
- Si Buscalibre carga ciertos campos (editorial, páginas, idioma) mediante JavaScript, solo Selenium los obtendrá.
- Los campos básicos (título, precio, ISBN) suelen estar en el HTML estático o en datos estructurados JSON-LD, por lo que los tres métodos deberían coincidir.

## 12. Conclusiones

1. **Cada herramienta tiene un caso de uso óptimo.** BeautifulSoup es ideal para extracciones simples y rápidas sobre HTML estático. Selenium es necesario cuando el sitio carga contenido con JavaScript. Scrapy es la mejor opción para proyectos de scraping a gran escala.

2. **La elección del método depende del sitio y del objetivo.** En el caso de Buscalibre, parte de la información del producto está disponible en el HTML estático (título, precio, ISBN via JSON-LD), pero la ficha técnica completa puede requerir renderizado JavaScript.

3. **El respeto a las condiciones de acceso es fundamental.** Se verificó `robots.txt` antes de realizar solicitudes, se usó un único acceso por método y no se intentó evadir ningún mecanismo de protección.

4. **Las métricas de éxito permiten evaluar objetivamente cada método.** La completitud mide cuántos campos se extrajeron, la concordancia valida la consistencia entre métodos, y la ejecución sin errores evalúa la robustez del código.

---

**Nota:** Los resultados dependen del estado del sitio al momento de la ejecución. Los precios, disponibilidad y estructura HTML pueden cambiar.

## 13. Fuentes y referencias

- Buscalibre Colombia. (s.f.). *Cien años de soledad (Edición conmemorativa de la RAE y la ASALE)*. Recuperado de https://www.buscalibre.com.co/libro-cien-anos-de-soledad-edicion-conmemorativa-de-la-rae-y-la-asale/9789585118010/p/52342100
- Buscalibre Colombia. (s.f.). *robots.txt*. Recuperado de https://www.buscalibre.com.co/robots.txt
- Richardson, L. (2023). *Beautiful Soup Documentation*. Recuperado de https://www.crummy.com/software/BeautifulSoup/bs4/doc/
- Selenium Project. (2024). *Selenium with Python*. Recuperado de https://selenium-python.readthedocs.io/
- Scrapy Project. (2024). *Scrapy Documentation*. Recuperado de https://docs.scrapy.org/en/latest/
- Python Software Foundation. (2024). *Requests: HTTP for Humans*. Recuperado de https://docs.python-requests.org/en/latest/